# SLAP-GRO — Phase 0: Data Audit & Reconciliation

This notebook provides an interactive inspection of all raw datasets, verifying schemas, delimiters, foreign keys, and known discrepancies between the literature setup and the supplied industrial dataset.

In [3]:
import sys
from pathlib import Path

# Ensure project root and src/ are in sys.path regardless of where the notebook kernel is launched
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

for path in [str(project_root), str(project_root / "src")]:
    if path not in sys.path:
        sys.path.insert(0, path)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from slap_gro.config.loader import load_data_contract
from slap_gro.data import (
    load_products,
    load_storage_locations,
    load_navigation_points,
    load_customer_orders,
    load_picking_waves,
    load_storage_matrix,
)

# Load data contract
contract = load_data_contract()
print(f"Project Root: {project_root}")
print(f"Data Contract Version: {contract.get('version')}")

ModuleNotFoundError: No module named 'slap_gro'

## 1. Load Summary Audit Table

In [5]:
# Resolve summary CSV path robustly from project root
summary_csv = project_root / "outputs" / "tables" / "data_audit_summary.csv"
if not summary_csv.exists():
    from scripts.audit_data import run_audit
    run_audit()

summary_df = pd.read_csv(summary_csv)
summary_df

ModuleNotFoundError: No module named 'scripts'

## 2. Inspect Products (`Product.csv`)

In [ ]:
prods = load_products()
print(f"Total unique products: {len(prods)}")
print("ABC Class Distribution:")
print(prods['ABCCOD'].value_counts())
prods.head()

## 3. Inspect Physical Storage Locations (`Storage_Location.csv`)

In [ ]:
locs = load_storage_locations()
print(f"Total locations: {len(locs)}")
print("Levels (z) distribution:")
print(locs['z'].value_counts().sort_index())

# Visualize 2D scatter of locations
plt.figure(figsize=(12, 6))
scatter = plt.scatter(locs['x'], locs['y'], c=locs['z'], cmap='viridis', s=10, alpha=0.7)
plt.colorbar(scatter, label='Level (z)')
plt.title('Warehouse Storage Locations (2D Projection with Level Color)')
plt.xlabel('X (meters)')
plt.ylabel('Y (meters)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 4. Inspect Support Navigation Points (`Support_Points_Navigation.csv`)

In [ ]:
nav = load_navigation_points()
print(f"Total navigation points: {len(nav)}")
nav.head(10)

## 5. Storage Policies Comparison (Random, Class-Based, Dedicated, Hybrid)

In [ ]:
rs = load_storage_matrix('random')
cb = load_storage_matrix('class_based')
ds = load_storage_matrix('dedicated')
hs = load_storage_matrix('hybrid')

print(f"Random Storage locations:     {len(rs)}")
print(f"Class-Based Storage locations:{len(cb)} (49 extra locations: aisles 25-26 + Q-26-20)")
print(f"Dedicated Storage locations:  {len(ds)} (48 extra locations: aisles 25-26)")
print(f"Hybrid Storage locations:     {len(hs)} (48 extra locations: aisles 25-26)")